# GINN: GARCH-informed neural network
Step 1: GARCH(1,1) and GJR-GARCH fitted by QMLE in torch, on training days only.
Data: last 2,500 trading days of Mkt-RF. Train = first 2,000 (2016-09-20 to 2024-08-30), validation = last 500.

In [41]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

In [42]:
d = pd.read_parquet('../../data/ff_daily_garchdata_long.parquet')
r_all = d.loc[d.series == 'Mkt-RF', 'ret'].to_numpy()[-2500:] * 100     # last 2,500 days, in percent
r_all = torch.tensor(r_all, dtype=torch.float64)

N_TRAIN = 2000
r_train = r_all[:N_TRAIN]                                                  # GARCH / GJR are fitted on these only
print(len(r_all), len(r_train))

2500 2000


## 1a. GARCH(1,1)
h(t) = ω + α·r²(t−1) + β·h(t−1).  Softmax over [α, β, leftover] keeps α + β < 1.

In [43]:
def unpack_garch(raw):
    omega = torch.nn.functional.softplus(raw[0]) * 0.1
    w = torch.softmax(torch.cat([raw[1:], torch.zeros(1, dtype=raw.dtype)]), dim=0)   # [alpha, beta, leftover]
    return omega, w[0], w[1]


def nll_garch(raw, r):
    omega, alpha, beta = unpack_garch(raw)
    h = r.var()
    total = torch.zeros((), dtype=r.dtype)
    for t in range(1, len(r)):
        h = omega + alpha * r[t - 1] ** 2 + beta * h
        total = total + torch.log(h) + r[t] ** 2 / h
    return 0.5 * total / (len(r) - 1)

## 1b. GJR-GARCH
h(t) = ω + (α + γ·1{r(t−1)<0})·r²(t−1) + β·h(t−1).
Softmax over [α, γ/2, β, leftover], then γ = 2·w[1], so α + γ/2 + β < 1 automatically.
Starting values: α ≈ 0.03, γ/2 ≈ 0.05, β ≈ 0.90, leftover ≈ 0.02, so raw = ln(target / leftover).

In [44]:
def unpack_gjr(raw):
    omega = torch.nn.functional.softplus(raw[0]) * 0.1
    w = torch.softmax(torch.cat([raw[1:], torch.zeros(1, dtype=raw.dtype)]), dim=0)   # [alpha, gamma/2, beta, leftover]
    return omega, w[0], 2 * w[1], w[2]


def nll_gjr(raw, r):
    omega, alpha, gamma, beta = unpack_gjr(raw)
    h = r.var()
    total = torch.zeros((), dtype=r.dtype)
    for t in range(1, len(r)):
        neg = (r[t - 1] < 0).to(r.dtype)                 # 1.0 after a down day, 0.0 otherwise
        h = omega + (alpha + gamma * neg) * r[t - 1] ** 2 + beta * h
        total = total + torch.log(h) + r[t] ** 2 / h
    return 0.5 * total / (len(r) - 1)

## 1c. One fitting function for both models
Same four-line loop as before. `raw0` is the starting point; a fresh copy is trained each call.

In [45]:
def fit(nll_fn, unpack_fn, raw0, r, steps=500, lr=0.05, every=100):
    raw = raw0.clone().requires_grad_(True)
    opt = torch.optim.Adam([raw], lr=lr)
    for step in range(steps + 1):
        opt.zero_grad()
        loss = nll_fn(raw, r)
        loss.backward()
        opt.step()
        if step % every == 0:
            print(f'Step {step} | Loss {loss.item():.5f} | params {[round(x.item(), 4) for x in unpack_fn(raw)]}')
    raw = raw.detach()
    return raw, nll_fn(raw, r).item()

In [46]:
raw_garch, loss_garch = fit(nll_garch, unpack_garch,
                            torch.tensor([-1.0, 1.39, 3.81], dtype=torch.float64), r_train)
print('GARCH  omega, alpha, beta =', [round(x.item(), 4) for x in unpack_garch(raw_garch)], '| loss', round(loss_garch, 5))
print('target (scipy):[0.0305, 0.1625, 0.8167] | loss 0.41328')

Step 0 | Loss 0.43162 | params [0.03, 0.0876, 0.8916]
Step 100 | Loss 0.41329 | params [0.0314, 0.1614, 0.8155]
Step 200 | Loss 0.41328 | params [0.0308, 0.1621, 0.8163]
Step 300 | Loss 0.41328 | params [0.0306, 0.1624, 0.8166]
Step 400 | Loss 0.41328 | params [0.0305, 0.1625, 0.8167]
Step 500 | Loss 0.41328 | params [0.0305, 0.1625, 0.8167]
GARCH  omega, alpha, beta = [0.0305, 0.1625, 0.8167] | loss 0.41328
target (scipy):            [0.0305, 0.1625, 0.8167] | loss 0.41328


In [48]:
raw_gjr, loss_gjr = fit(nll_gjr, unpack_gjr,
                        torch.tensor([-1.0, 0.405, 0.916, 3.807], dtype=torch.float64), r_train)
om, a, g, b = [x.item() for x in unpack_gjr(raw_gjr)]
print('GJR  omega, alpha, gamma, beta =', [round(v, 4) for v in (om, a, g, b)], '| loss', round(loss_gjr, 5),
      '| persistence', round(a + g / 2 + b, 4))
print('target (scipy):[0.0341, 0.0581, 0.2244, 0.8146] | loss 0.39633 | persistence 0.985')

Step 0 | Loss 0.41331 | params [0.03, 0.0328, 0.1094, 0.8916]
Step 100 | Loss 0.39643 | params [0.0355, 0.0555, 0.2156, 0.8155]
Step 200 | Loss 0.39635 | params [0.0348, 0.0567, 0.2199, 0.8151]
Step 300 | Loss 0.39634 | params [0.0344, 0.0574, 0.2221, 0.8149]
Step 400 | Loss 0.39633 | params [0.0342, 0.0577, 0.2233, 0.8147]
Step 500 | Loss 0.39633 | params [0.0341, 0.0579, 0.2239, 0.8147]
GJR  omega, alpha, gamma, beta = [0.0341, 0.0579, 0.2239, 0.8147] | loss 0.39633 | persistence 0.9845
target (scipy):                 [0.0341, 0.0581, 0.2244, 0.8146] | loss 0.39633 | persistence 0.985


## 2. Frozen GJR variance path for all 2,500 days
Parameters fixed at the training fit. `@torch.no_grad()` means no gradients: the GJR branch is frozen.
Check: the loss on training days must equal the fitted GJR loss.

In [49]:
@torch.no_grad()
def gjr_path(raw, r, h0):
    omega, alpha, gamma, beta = unpack_gjr(raw)
    h = torch.empty_like(r)
    h[0] = h0
    for t in range(1, len(r)):
        neg = (r[t - 1] < 0).to(r.dtype)
        h[t] = omega + (alpha + gamma * neg) * r[t - 1] ** 2 + beta * h[t - 1]
    return h


h_gjr = gjr_path(raw_gjr, r_all, r_train.var()) # h(t) uses information up to day t-1 only

check = 0.5 * torch.mean(torch.log(h_gjr[1:N_TRAIN]) + r_all[1:N_TRAIN] ** 2 / h_gjr[1:N_TRAIN])
print('loss from path on train days:', round(check.item(), 5), '| fitted GJR loss:', round(loss_gjr, 5))

loss from path on train days: 0.39633 | fitted GJR loss: 0.39633


## 3. Features, 3 per day: r, |r|, log h_GJR
Standardised with training-day mean and std only (days 0-1999).

In [51]:
F_raw = torch.stack([r_all, r_all.abs(), torch.log(h_gjr)], dim=1)    # (2500, 3)
mu = F_raw[:N_TRAIN].mean(dim=0)
sd = F_raw[:N_TRAIN].std(dim=0)
F = ((F_raw - mu) / sd).to(torch.float32)

print('train mean', F[:N_TRAIN].mean(dim=0).numpy().round(3), '| train std', F[:N_TRAIN].std(dim=0).numpy().round(3))
print('valid mean', F[N_TRAIN:].mean(dim=0).numpy().round(3), '| valid std', F[N_TRAIN:].std(dim=0).numpy().round(3))

train mean [-0. -0. -0.] | train std [1. 1. 1.]
valid mean [ 0.002 -0.083 -0.031] | valid std [0.88  0.862 0.775]


## 4. Windows and targets
Target day t uses features of days t-22 … t-1. Train = targets t < 2000, validation = t >= 2000.
The asserts prove each window ends on day t-1 and never contains day t.

In [52]:
L = 22
targets = torch.arange(L, len(r_all))                    # t = 22 … 2499
X = torch.stack([F[t - L:t] for t in targets])           # (2478, 22, 3)
y = r_all[targets].to(torch.float32)                     # return on the target day
hg = h_gjr[targets].to(torch.float32)                    # GJR variance for the target day

for i in (0, 1000, len(targets) - 1):
    t = targets[i].item()
    assert torch.equal(X[i, -1], F[t - 1]) and torch.equal(X[i, 0], F[t - L])

is_tr = targets < N_TRAIN
Xtr, ytr, hgtr = X[is_tr], y[is_tr], hg[is_tr]
Xva, yva, hgva = X[~is_tr], y[~is_tr], hg[~is_tr]
print('X', tuple(X.shape), '| train', tuple(Xtr.shape), '| validation', tuple(Xva.shape))

X (2478, 22, 3) | train (1978, 22, 3) | validation (500, 22, 3)


## 5. The GINN
h_GINN = h_GJR × exp(g),  g = Linear(LSTM(window)).  The last layer starts at zero, so the GINN starts as GJR.

In [53]:
class GINN(nn.Module):
    def __init__(self, n_features=3, hidden=16):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, x, h_base):
        out, _ = self.lstm(x)                         # (N, 22, hidden)
        g = self.head(out[:, -1, :]).squeeze(-1)      # (N,)
        return h_base * torch.exp(g)


def qmle(h, r):
    return 0.5 * torch.mean(torch.log(h) + r ** 2 / h)


torch.manual_seed(0)
model = GINN()
print(model)
print('trainable parameters:', sum(p.numel() for p in model.parameters()))

GINN(
  (lstm): LSTM(3, 16, batch_first=True)
  (head): Linear(in_features=16, out_features=1, bias=True)
)
trainable parameters: 1361


## 6. Checkpoint before training
With g = 0 the GINN must reproduce GJR exactly, on both train and validation windows.

In [54]:
with torch.no_grad():
    h_init = model(Xtr, hgtr)
print('GINN == GJR at start:', torch.allclose(h_init, hgtr))
print('train loss  GINN', round(qmle(h_init, ytr).item(), 5), '| GJR', round(qmle(hgtr, ytr).item(), 5))

GINN == GJR at start: True
train loss  GINN 0.39911 | GJR 0.39911


## 7. Training with validation and early stopping
Full batch (all 1,978 training windows each step). Validation every 10 epochs; keep the best weights.
`clip_grad_norm_` caps very large gradients, which keeps LSTM training stable after crash days.

In [55]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS, PATIENCE = 2000, 300
best_val, best_state, best_epoch = float('inf'), copy.deepcopy(model.state_dict()), 0
gjr_val = qmle(hgva, yva).item()

for epoch in range(1, EPOCHS + 1):
    model.train()
    opt.zero_grad()
    loss = qmle(model(Xtr, hgtr), ytr)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    opt.step()

    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            val = qmle(model(Xva, hgva), yva).item()
        if val < best_val:
            best_val, best_state, best_epoch = val, copy.deepcopy(model.state_dict()), epoch
        if epoch % 100 == 0:
            print(f'epoch {epoch:4d} | train {loss.item():.5f} | val {val:.5f} | best val {best_val:.5f} @ {best_epoch} | GJR val {gjr_val:.5f}')
        if epoch - best_epoch >= PATIENCE:
            print(f'stopped at epoch {epoch}: no validation improvement for {PATIENCE} epochs')
            break

model.load_state_dict(best_state)

epoch  100 | train 0.38465 | val 0.38556 | best val 0.38556 @ 100 | GJR val 0.40815
epoch  200 | train 0.33467 | val 0.41986 | best val 0.38283 @ 110 | GJR val 0.40815
epoch  300 | train 0.30829 | val 0.46365 | best val 0.38283 @ 110 | GJR val 0.40815
epoch  400 | train 0.28762 | val 0.48665 | best val 0.38283 @ 110 | GJR val 0.40815
stopped at epoch 410: no validation improvement for 300 epochs


<All keys matched successfully>

## 8. Verdict on the validation days
Loss per day for GJR and GINN, the difference, and a Diebold-Mariano t-statistic
(Newey-West standard error, 5 lags). Negative difference = GINN better.

In [56]:
model.eval()
with torch.no_grad():
    h_va = model(Xva, hgva)
    g_va = torch.log(h_va / hgva)

loss_gjr_day = 0.5 * (torch.log(hgva) + yva ** 2 / hgva)
loss_ginn_day = 0.5 * (torch.log(h_va) + yva ** 2 / h_va)
dd = (loss_ginn_day - loss_gjr_day).double().numpy()


def dm_tstat(d, lags=5):
    n, u = len(d), d - d.mean()
    lrv = np.mean(u * u) + 2 * sum((1 - k / (lags + 1)) * np.mean(u[k:] * u[:-k]) for k in range(1, lags + 1))
    return d.mean() / np.sqrt(lrv / n)


print(f'best epoch: {best_epoch}')
print(f'validation loss  GJR  {loss_gjr_day.mean().item():.5f}')
print(f'validation loss  GINN {loss_ginn_day.mean().item():.5f}')
print(f'difference (GINN - GJR) {dd.mean():+.5f} | DM t-stat {dm_tstat(dd):+.2f}  (|t| > 1.96 = significant at 5%)')
print(f'correction g on validation: mean {g_va.mean().item():+.3f}, std {g_va.std().item():.3f}, '
      f'min {g_va.min().item():+.3f}, max {g_va.max().item():+.3f}')

best epoch: 110
validation loss  GJR  0.40815
validation loss  GINN 0.38283
difference (GINN - GJR) -0.02532 | DM t-stat -2.80  (|t| > 1.96 = significant at 5%)
correction g on validation: mean -0.060, std 0.226, min -0.692, max +0.540
